In [0]:
%pip install SQLAlchemy psycopg2-binary python-dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from sqlalchemy import create_engine, text
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, col, lit, monotonically_increasing_id, right
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()


In [0]:
postgres_driver = "/Users/s.ekundayoapi@gmail.com/databricks-etl/postgresql-42.7.13.jar"
spark = (
    SparkSession.builder.appName("Nuga Bank ETL")
    .config("spark.jars", postgres_driver)
    .getOrCreate()
)

In [0]:
nugabank_df = spark.read.csv("/Volumes/workspace/default/nugabank/nuga_bank_transactions.csv", header=True, inferSchema=True)
nugabank_df.show(10)

In [0]:
nugabank_df_clean = nugabank_df
nugabank_df_clean = nugabank_df_clean.toDF(*[c.lower() for c in nugabank_df.columns])

In [0]:
nugabank_df_clean.printSchema()

In [0]:
# How to fill up missing values
nugabank_df_clean = nugabank_df.fillna({
    'customer_name' : 'unknown',
    'customer_address' : 'unknown',
    'customer_city' : 'unknown',
    'customer_state' : 'unknown',
    'customer_country' : 'unknown',
    'company' : 'unknown',
    'job_title' : 'unknown',
    'email' : 'unknown',
    'phone_number' : 'unknown',
    'credit_card_number' : 0,
    'iban' : 'unknown',
    'currency_code' : 'unknown',
    'random_number' : 0.0,
    'category' : 'unknown',
    'group' : 'unknown',
    'is_active' : 'unknown',
    'description' : 'unknown',
    'gender' : 'unknown',
    'marital_status' : 'unknown'
})


In [0]:
# Drop rows where last_updated is null
nugabank_df_clean = nugabank_df_clean.na.drop(subset=['last_updated'])

In [0]:
# transaction table
transaction = nugabank_df_clean.select('transaction_date','amount','transaction_type') 


In [0]:
# Adding the transaction_id column
transaction = transaction.withColumn('transaction_id', monotonically_increasing_id())


In [0]:
# Customer table
customer = nugabank_df_clean.select(
    "customer_name",
    "customer_address",
    "customer_city",
    "customer_state",
    "customer_country",
    "email",
    "phone_number",
).distinct()

# add id column
customer = customer.withColumn("customer_id", monotonically_increasing_id())

# reorder the table
customer = customer.select(
    "customer_id",
    "customer_name",
    "customer_address",
    "customer_city",
    "customer_state",
    "customer_country",
    "email",
    "phone_number",
)

In [0]:
# employee table
employees = nugabank_df_clean.select('company', 'job_title', 'gender', 'marital_status').distinct()

# add id column
employee = employees.withColumn('employee_id', monotonically_increasing_id())

# re-order the dataframe
employee = employee.select('employee_id', 'company', 'job_title', 'gender', 'marital_status')


In [0]:
# fact_table

fact_table = (
    nugabank_df_clean.join(
        customer,
        [
            "customer_name",
            "customer_address",
            "customer_city",
            "customer_state",
            "customer_country",
            "email",
            "phone_number",
        ],
        "left",
    )
    .join(transaction, ["transaction_date", "amount", "transaction_type"], "left")
    .join(employee, ["company", "job_title", "gender", "marital_status"], "left")
    .select(
        "transaction_id",
        "customer_id",
        "employee_id",
        "credit_card_number",
        "iban",
        "currency_code",
        "random_number",
        "category",
        "group",
        "is_active",
        "last_updated",
        "description",
    )
)

In [0]:
# Mask sensitive values but keep their final four characters.
analytics_fact_table = (
    fact_table
    .withColumn(
        "IBAN",
        concat(lit("********"), right("IBAN", lit(4))),
    )
    .withColumn(
        "Credit_Card_Number",
        concat(
            lit("********"),
            right(col("Credit_Card_Number").cast("string"), lit(4)),
        ),
    )
)

In [0]:
# output the transformed data as csv
transaction.repartition(1).write.mode("overwrite").option("header", "true").csv(
    r"/Volumes/workspace/default/nugabank/transaction"
)
customer.repartition(1).write.mode("overwrite").option("header", "true").csv(
    r"/Volumes/workspace/default/nugabank/customer"
)
employee.repartition(1).write.mode("overwrite").option("header", "true").csv(
    r"/Volumes/workspace/default/nugabank/employee"
)
fact_table.repartition(1).write.mode("overwrite").option("header", "true").csv(
    r"/Volumes/workspace/default/nugabank/fact_table"
)

In [0]:
# Create a database in PostgreSQL and store the data in a table
# Define the database connection parameters
# Prefer environment variables if available, otherwise fall back to local defaults.
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "password")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "databricksNB_db")
db_ssl_mode = os.getenv("DB_SSL_MODE")


In [0]:
# Using Unity Catalog for data storage (native Databricks)
# Unity Catalog provides Delta Lake tables with built-in:
# - ACID transactions
# - Time travel and versioning
# - Fine-grained access control
# - Data lineage tracking

print("Data will be saved to Unity Catalog Delta tables")
print("Location: workspace.nugabank schema")
print("\nNote: Original PostgreSQL connection was not possible due to")
print("network restrictions between Databricks Serverless and Supabase.")

In [0]:
# Note: PostgreSQL/Supabase connection was not possible from Databricks Serverless
# Using Unity Catalog Delta tables instead (see Cell 22)
# Kept environment variables for reference


In [0]:
# Unity Catalog manages schema and tables automatically
# Tables are created as Delta format with full ACID support
# No manual DDL needed - schema is inferred from DataFrames

print("Using Unity Catalog Delta tables:")
print("- workspace.nugabank.transaction")
print("- workspace.nugabank.customer")
print("- workspace.nugabank.employee")
print("- workspace.nugabank.fact_table")
print("\nTables will be created automatically in Cell 22")

In [0]:
# PostgreSQL JDBC connection was not possible from Databricks Serverless
# Using Unity Catalog instead (see Cell 22)
print("Skipping PostgreSQL JDBC setup - using Unity Catalog")

In [0]:
# Save to Unity Catalog Delta tables (native Databricks storage)
catalog_name = "workspace"
schema_name = "nugabank"

# Create schema for the bank data
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✓ Schema {catalog_name}.{schema_name} ready")

# Write tables to Unity Catalog with Delta format
transaction.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.transaction")
print(f"✓ Saved transaction table ({transaction.count()} rows)")

customer.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.customer")
print(f"✓ Saved customer table ({customer.count()} rows)")

employee.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.employee")
print(f"✓ Saved employee table ({employee.count()} rows)")

analytics_fact_table.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.fact_table")
print(f"✓ Saved fact_table ({analytics_fact_table.count()} rows)")

print(f"\n🎉 All tables saved to Unity Catalog!")
print(f"\nYou can now query them with SQL:")
print(f"  SELECT * FROM {catalog_name}.{schema_name}.transaction LIMIT 10;")
print(f"  SELECT * FROM {catalog_name}.{schema_name}.customer LIMIT 10;")
print(f"  SELECT * FROM {catalog_name}.{schema_name}.employee LIMIT 10;")
print(f"  SELECT * FROM {catalog_name}.{schema_name}.fact_table LIMIT 10;")

In [0]:
# Put the four Spark DataFrames in one dictionary.
tables = {
    "customer": customer,
    "employee": employee,
    "transaction": transaction,
    "fact_table": fact_table,
}

for table_name, dataframe in tables.items():
    dataframe.createOrReplaceTempView(table_name)
    print(f"Created Spark view: {table_name}")

In [0]:
# Query the transaction table using Spark SQL
# Preview transactions
spark.sql("""
    SELECT
        transaction_id,
        transaction_date,
        amount,
        transaction_type
    FROM transaction
    LIMIT 10
""").show(truncate=False)

In [0]:
# Total and average transaction amount by type
spark.sql("""
    SELECT
        transaction_type,
        COUNT(*) AS transaction_count,
        ROUND(SUM(Amount), 2) AS total_amount,
        ROUND(AVG(Amount), 2) AS average_amount
    FROM transaction
    GROUP BY transaction_type
    ORDER BY total_amount DESC
""").show(truncate=False)

In [0]:
# Top 20 transactions by date - Most recent transactions
spark.sql("""
    SELECT
        t.transaction_id,
        t.transaction_date,
        t.transaction_type,
        t.amount,
        c.customer_name,
        c.customer_city,
        c.customer_country
    FROM transaction AS t
    INNER JOIN fact_table AS f
        ON t.transaction_id = f.transaction_id
    INNER JOIN customer AS c
        ON f.customer_id = c.customer_id
    ORDER BY t.transaction_date DESC
    LIMIT 20
""").show(truncate=False)